# Dynamic Networks — a catalog of network-reconfiguring rules

_Investigation `dynamic-networks` — coder reproduction notebook._

**Question.** How do different local rules reshape a network over time, and can a single process-bigraph process — driven by pluggable reactions — reproduce the canonical families of network dynamics, each with a clear visualization of the network evolving?

A catalog investigation: eight studies, each a different network-reconfiguring rule implemented as a process-bigraph reaction on a shared graph substrate, each with an evolving-network animation and GIF.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-dynamic-networks/viva-dynamic-networks').is_dir():
    REPO = Path('/home/runner/work/viva-dynamic-networks/viva-dynamic-networks')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_dynamic_networks.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Erdős–Rényi random growth (`01-erdos-renyi-random-growth`)

**Question.** What topology emerges when edges form purely at random, with no preference?

**Objective.** Run the erdos_renyi reaction and show how it reconfigures the network, measuring its signature structural observables.

**Hypothesis.** Random attachment yields a homogeneous Poisson degree distribution with no hubs — the null baseline for every structured rule that follows.

**Claim.** Random growth produces a homogeneous, hub-free network (Poisson-like degrees) — the null baseline the structured rules are read against.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_dynamic_networks.composites.family_a.erdos_renyi` | 0 | p=0.08, max_nodes=60.0, seed=7.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_a.erdos_renyi`**

Random growth (Erdős–Rényi): a node arrives each step and links to each existing node with probability p. Poisson degrees; the null baseline every other rule is read against.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_a_erdos_renyi = {
    'p'        : 0.08,   # edge probability
    'max_nodes': 60.0,   # growth cap
    'seed'     : 0.0,   # RNG seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 01-erdos-renyi-random-growth ===
STUDY = '01-erdos-renyi-random-growth'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_a_erdos_renyi if 'ov_viva_dynamic_networks_composites_family_a_erdos_renyi' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_a_erdos_renyi = materialize_composite('viva_dynamic_networks.composites.family_a.erdos_renyi', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_a_erdos_renyi['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_dynamic_networks.composites.family_a.erdos_renyi) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**network-evolution (live)**


In [ ]:
# network-evolution (live)
show_viz(_render_one('local:NetworkEvolution', {'title': 'Erdős–Rényi random growth', 'layout': 'spring', 'color_by': 'degree', 'subtitle': 'node arrives each step, links to each existing node w.p. p', 'composite': 'viva_dynamic_networks.composites.family_a.erdos_renyi', 'n_steps': 60, 'render_mode': 'end'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_dynamic_networks_composites_family_a_erdos_renyi if 'ov_viva_dynamic_networks_composites_family_a_erdos_renyi' in dir() else None)))

**network-evolution (animation)**


In [ ]:
# network-evolution (animation)
show_viz(_render_one('gif:erdos_renyi.gif', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| er-giant-component _(quantitative)_ | kind=flat observable=largest_component_fraction reduce=last | op at_least value 0.9 |
| er-no-dominant-hub _(quantitative)_ | kind=flat observable=max_degree reduce=last | op at_most value 16 |


## Study: Barabási–Albert preferential attachment (`02-preferential-attachment`)

**Question.** Does degree-proportional attachment produce scale-free hubs?

**Objective.** Run the preferential_attachment reaction and show how it reconfigures the network, measuring its signature structural observables.

**Hypothesis.** Linking new nodes preferentially to high-degree nodes yields a power-law degree distribution with a few dominant hubs.

**Claim.** Degree-proportional attachment produces scale-free hubs with a power-law degree tail — unlike random growth.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_dynamic_networks.composites.family_a.preferential_attachment` | 0 | m=2.0, max_nodes=60.0, seed=7.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_a.preferential_attachment`**

Barabási–Albert: a node arrives each step with m edges to existing nodes chosen with probability ∝ degree — scale-free hubs emerge ('rich get richer').


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_a_preferential_attachment = {
    'm'        : 2.0,   # edges per new node
    'max_nodes': 60.0,   # growth cap
    'seed'     : 0.0,   # RNG seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 02-preferential-attachment ===
STUDY = '02-preferential-attachment'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_a_preferential_attachment if 'ov_viva_dynamic_networks_composites_family_a_preferential_attachment' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_a_preferential_attachment = materialize_composite('viva_dynamic_networks.composites.family_a.preferential_attachment', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_a_preferential_attachment['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_dynamic_networks.composites.family_a.preferential_attachment) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**network-evolution (live)**


In [ ]:
# network-evolution (live)
show_viz(_render_one('local:NetworkEvolution', {'title': 'Barabási–Albert preferential attachment', 'layout': 'spring', 'color_by': 'degree', 'subtitle': "new nodes attach to existing nodes ∝ degree ('rich get richer')", 'composite': 'viva_dynamic_networks.composites.family_a.preferential_attachment', 'n_steps': 58, 'render_mode': 'end'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_dynamic_networks_composites_family_a_preferential_attachment if 'ov_viva_dynamic_networks_composites_family_a_preferential_attachment' in dir() else None)))

**network-evolution (animation)**


In [ ]:
# network-evolution (animation)
show_viz(_render_one('gif:preferential_attachment.gif', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| ba-hub-emerges _(quantitative)_ | kind=flat observable=max_degree reduce=last | op at_least value 10 |
| ba-connected _(quantitative)_ | kind=flat observable=largest_component_fraction reduce=last | op at_least value 0.99 |


## Study: Watts–Strogatz small-world rewiring (`03-small-world-rewiring`)

**Question.** How does random rewiring of a ring lattice change its structure?

**Objective.** Run the small_world_rewire reaction and show how it reconfigures the network, measuring its signature structural observables.

**Hypothesis.** Rewiring a small fraction of lattice edges collapses path length while clustering stays high — the small-world regime — without changing node or edge counts.

**Claim.** Random rewiring of a ring lattice preserves edge count while shortcuts collapse path length — producing the small-world regime.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_dynamic_networks.composites.family_a.small_world_rewire` | 0 | p=0.14, n=28.0, k=4.0, seed=7.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_a.small_world_rewire`**

Watts–Strogatz: on a ring lattice, rewire a random edge with probability p each step — a tidy ring dissolves into a small world (high clustering, short paths).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_a_small_world_rewire = {
    'p'   : 0.12,   # per-step rewiring probability
    'n'   : 28.0,   # number of nodes
    'k'   : 4.0,   # ring lattice degree
    'seed': 0.0,   # RNG seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 03-small-world-rewiring ===
STUDY = '03-small-world-rewiring'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_a_small_world_rewire if 'ov_viva_dynamic_networks_composites_family_a_small_world_rewire' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_a_small_world_rewire = materialize_composite('viva_dynamic_networks.composites.family_a.small_world_rewire', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_a_small_world_rewire['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_dynamic_networks.composites.family_a.small_world_rewire) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**network-evolution (live)**


In [ ]:
# network-evolution (live)
show_viz(_render_one('local:NetworkEvolution', {'title': 'Watts–Strogatz small-world rewiring', 'layout': 'circular', 'color_by': 'degree', 'subtitle': 'a ring lattice dissolves into a small world as random rewires accumulate', 'composite': 'viva_dynamic_networks.composites.family_a.small_world_rewire', 'n_steps': 45, 'render_mode': 'end'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_dynamic_networks_composites_family_a_small_world_rewire if 'ov_viva_dynamic_networks_composites_family_a_small_world_rewire' in dir() else None)))

**network-evolution (animation)**


In [ ]:
# network-evolution (animation)
show_viz(_render_one('gif:small_world_rewire.gif', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| sw-edge-count-conserved _(invariant)_ | kind=flat observable=n_edges reduce=last | op equals value 56 tolerance 0 |
| sw-node-count-conserved _(invariant)_ | kind=flat observable=n_nodes reduce=last | op equals value 28 tolerance 0 |


## Study: Bianconi–Barabási fitness attachment (`04-fitness-condensation`)

**Question.** Can an intrinsic node fitness let a latecomer overtake older, higher-degree nodes?

**Objective.** Run the fitness_attachment reaction and show how it reconfigures the network, measuring its signature structural observables.

**Hypothesis.** Weighting attachment by fitness×degree still produces hubs, but the leading hub need not be the oldest node — fitness breaks the pure first-mover advantage.

**Claim.** Fitness-weighted attachment still builds hubs, but a high-fitness latecomer can overtake older nodes — unlike pure Barabási–Albert.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_dynamic_networks.composites.family_a.fitness_attachment` | 0 | m=2.0, max_nodes=60.0, seed=7.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_a.fitness_attachment`**

Bianconi–Barabási: preferential attachment weighted by an intrinsic per-node fitness η — a fit latecomer can overtake older hubs.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_a_fitness_attachment = {
    'm'        : 2.0,   # edges per new node
    'max_nodes': 60.0,   # growth cap
    'seed'     : 0.0,   # RNG seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 04-fitness-condensation ===
STUDY = '04-fitness-condensation'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_a_fitness_attachment if 'ov_viva_dynamic_networks_composites_family_a_fitness_attachment' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_a_fitness_attachment = materialize_composite('viva_dynamic_networks.composites.family_a.fitness_attachment', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_a_fitness_attachment['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_dynamic_networks.composites.family_a.fitness_attachment) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**network-evolution (live)**


In [ ]:
# network-evolution (live)
show_viz(_render_one('local:NetworkEvolution', {'title': 'Bianconi–Barabási fitness attachment', 'layout': 'spring', 'color_by': 'degree', 'subtitle': 'attachment ∝ fitness × degree — a fit latecomer can overtake old hubs', 'composite': 'viva_dynamic_networks.composites.family_a.fitness_attachment', 'n_steps': 58, 'render_mode': 'end'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_dynamic_networks_composites_family_a_fitness_attachment if 'ov_viva_dynamic_networks_composites_family_a_fitness_attachment' in dir() else None)))

**network-evolution (animation)**


In [ ]:
# network-evolution (animation)
show_viz(_render_one('gif:fitness_attachment.gif', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| fit-hub-emerges _(quantitative)_ | kind=flat observable=max_degree reduce=last | op at_least value 10 |
| fit-connected _(quantitative)_ | kind=flat observable=largest_component_fraction reduce=last | op at_least value 0.99 |


## Study: Duplication–divergence (biological growth) (`05-duplication-divergence`)

**Question.** Does genome-style duplication-and-divergence reproduce protein-network topology?

**Objective.** Run the duplication_divergence reaction and show how it reconfigures the network, measuring its signature structural observables.

**Hypothesis.** Copying a random node's neighbourhood and pruning edges yields a heavy-tailed, locally clustered network like observed protein-interaction graphs.

**Claim.** Copy-then-diverge growth reproduces the heavy-tailed, locally clustered topology seen in protein-interaction networks.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_dynamic_networks.composites.family_a.duplication_divergence` | 0 | q=0.55, p=0.1, max_nodes=60.0, seed=7.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_a.duplication_divergence`**

Duplication–divergence: a new node copies a random node's edges, retaining each with probability q (optional parent link with probability p) — reproduces protein-network topology.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_a_duplication_divergence = {
    'q'        : 0.5,   # edge-retention probability
    'p'        : 0.1,   # parent-link probability
    'max_nodes': 60.0,   # growth cap
    'seed'     : 0.0,   # RNG seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 05-duplication-divergence ===
STUDY = '05-duplication-divergence'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_a_duplication_divergence if 'ov_viva_dynamic_networks_composites_family_a_duplication_divergence' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_a_duplication_divergence = materialize_composite('viva_dynamic_networks.composites.family_a.duplication_divergence', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_a_duplication_divergence['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_dynamic_networks.composites.family_a.duplication_divergence) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**network-evolution (live)**


In [ ]:
# network-evolution (live)
show_viz(_render_one('local:NetworkEvolution', {'title': 'Duplication–divergence (biological growth)', 'layout': 'spring', 'color_by': 'degree', 'subtitle': "a new node copies a neighbour's edges, then sheds each with probability 1−q", 'composite': 'viva_dynamic_networks.composites.family_a.duplication_divergence', 'n_steps': 57, 'render_mode': 'end'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_dynamic_networks_composites_family_a_duplication_divergence if 'ov_viva_dynamic_networks_composites_family_a_duplication_divergence' in dir() else None)))

**network-evolution (animation)**


In [ ]:
# network-evolution (animation)
show_viz(_render_one('gif:duplication_divergence.gif', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| dd-hub-forms _(quantitative)_ | kind=flat observable=max_degree reduce=last | op at_least value 6 |
| dd-clustered _(quantitative)_ | kind=flat observable=clustering reduce=last | op at_least value 0.1 |


## Study: Forest-fire growth & densification (`06-forest-fire-densification`)

**Question.** Does recursive forest-fire burning densify the network over time?

**Objective.** Run the forest_fire reaction and show how it reconfigures the network, measuring its signature structural observables.

**Hypothesis.** Because each arrival can ignite many links, the edge count grows super-linearly in the node count — the network densifies and its mean degree rises.

**Claim.** Forest-fire growth densifies the network: mean degree rises as the graph grows, reproducing the densification law.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_dynamic_networks.composites.family_a.forest_fire` | 0 | p_forward=0.4, max_nodes=60.0, max_links=12.0, seed=7.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_a.forest_fire`**

Forest-fire growth: a new node burns recursively outward from a random ambassador — densification and shrinking diameter.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_a_forest_fire = {
    'p_forward': 0.37,   # forward burn probability
    'max_nodes': 60.0,   # growth cap
    'max_links': 12.0,   # max links per arrival
    'seed'     : 0.0,   # RNG seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 06-forest-fire-densification ===
STUDY = '06-forest-fire-densification'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_a_forest_fire if 'ov_viva_dynamic_networks_composites_family_a_forest_fire' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_a_forest_fire = materialize_composite('viva_dynamic_networks.composites.family_a.forest_fire', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_a_forest_fire['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_dynamic_networks.composites.family_a.forest_fire) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**network-evolution (live)**


In [ ]:
# network-evolution (live)
show_viz(_render_one('local:NetworkEvolution', {'title': 'Forest-fire growth & densification', 'layout': 'spring', 'color_by': 'degree', 'subtitle': "a new node 'burns' recursively outward, linking to a sampled subset of neighbours", 'composite': 'viva_dynamic_networks.composites.family_a.forest_fire', 'n_steps': 57, 'render_mode': 'end'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_dynamic_networks_composites_family_a_forest_fire if 'ov_viva_dynamic_networks_composites_family_a_forest_fire' in dir() else None)))

**network-evolution (animation)**


In [ ]:
# network-evolution (animation)
show_viz(_render_one('gif:forest_fire.gif', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| ff-densification _(quantitative)_ | kind=flat observable=mean_degree reduce=last | op at_least value 3.0 |
| ff-connected _(quantitative)_ | kind=flat observable=largest_component_fraction reduce=last | op at_least value 0.99 |


## Study: Adaptive coevolution (opinions + rewiring) (`07-adaptive-coevolution`)

**Question.** When topology and node state co-adapt, does the network organise into like-minded communities?

**Objective.** Run the adaptive_rewire reaction and show how it reconfigures the network, measuring its signature structural observables.

**Hypothesis.** Letting nodes rewire away from differing neighbours and toward similar ones (while opinions also spread) raises assortativity and segregates the two opinions — the coevolution bridge between Family A and dynamics-on-networks.

**Claim.** Coevolving opinions and rewiring self-organise the network into like-minded clusters — topology and state adapt to each other.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_dynamic_networks.composites.family_a.adaptive_rewire` | 0 | w=0.65, n=44.0, k=4.0, seed=7.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_a.adaptive_rewire`**

Gross–Blasius coevolution: opinions spread voter-style while, with rate w, nodes cut ties to differing neighbours and reconnect to like-minded ones — topology and state co-adapt.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_a_adaptive_rewire = {
    'w'   : 0.6,   # rewiring rate
    'n'   : 40.0,   # number of nodes
    'k'   : 4.0,   # ring lattice degree
    'seed': 0.0,   # RNG seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 07-adaptive-coevolution ===
STUDY = '07-adaptive-coevolution'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_a_adaptive_rewire if 'ov_viva_dynamic_networks_composites_family_a_adaptive_rewire' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_a_adaptive_rewire = materialize_composite('viva_dynamic_networks.composites.family_a.adaptive_rewire', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_a_adaptive_rewire['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_dynamic_networks.composites.family_a.adaptive_rewire) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**network-evolution (live)**


In [ ]:
# network-evolution (live)
show_viz(_render_one('local:NetworkEvolution', {'title': 'Adaptive coevolution (opinions + rewiring)', 'layout': 'spring', 'color_by': 'state', 'subtitle': 'opinions spread while nodes cut ties to the unlike and bond with the alike', 'composite': 'viva_dynamic_networks.composites.family_a.adaptive_rewire', 'n_steps': 70, 'render_mode': 'end'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_dynamic_networks_composites_family_a_adaptive_rewire if 'ov_viva_dynamic_networks_composites_family_a_adaptive_rewire' in dir() else None)))

**network-evolution (animation)**


In [ ]:
# network-evolution (animation)
show_viz(_render_one('gif:adaptive_rewire.gif', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| ad-edge-count-conserved _(invariant)_ | kind=flat observable=n_edges reduce=last | op equals value 88 tolerance 0 |
| ad-node-count-conserved _(invariant)_ | kind=flat observable=n_nodes reduce=last | op equals value 44 tolerance 0 |


## Study: Activity-driven temporal network (`08-activity-driven-temporal`)

**Question.** What does an activity-driven contact sequence look like, where edges last only one time window?

**Objective.** Run the activity_driven reaction and show how it reconfigures the network, measuring its signature structural observables.

**Hypothesis.** With edges reset every step and nodes firing in proportion to their activity, each instantaneous graph is sparse and blinking, while the time-integrated degree follows the activity distribution.

**Claim.** Activity-driven firing produces a sparse, blinking temporal network — each snapshot is instantaneous, unlike the accreting Family-A graphs.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_dynamic_networks.composites.family_a.activity_driven` | 0 | m=1.0, n=44.0, seed=7.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_a.activity_driven`**

Perra activity-driven temporal network: each step all edges reset, then each node fires with probability ∝ its activity and links to m random others — a blinking contact sequence.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_a_activity_driven = {
    'm'   : 1.0,   # edges per active node
    'n'   : 40.0,   # number of nodes
    'seed': 0.0,   # RNG seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 08-activity-driven-temporal ===
STUDY = '08-activity-driven-temporal'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_a_activity_driven if 'ov_viva_dynamic_networks_composites_family_a_activity_driven' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_a_activity_driven = materialize_composite('viva_dynamic_networks.composites.family_a.activity_driven', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_a_activity_driven['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_dynamic_networks.composites.family_a.activity_driven) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**network-evolution (live)**


In [ ]:
# network-evolution (live)
show_viz(_render_one('local:NetworkEvolution', {'title': 'Activity-driven temporal network', 'layout': 'spring', 'color_by': 'degree', 'subtitle': 'every step all edges reset; each node fires ∝ its activity, linking to m others', 'composite': 'viva_dynamic_networks.composites.family_a.activity_driven', 'n_steps': 50, 'render_mode': 'end'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_dynamic_networks_composites_family_a_activity_driven if 'ov_viva_dynamic_networks_composites_family_a_activity_driven' in dir() else None)))

**network-evolution (animation)**


In [ ]:
# network-evolution (animation)
show_viz(_render_one('gif:activity_driven.gif', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| adn-node-count-conserved _(invariant)_ | kind=flat observable=n_nodes reduce=last | op equals value 44 tolerance 0 |
| adn-sparse-snapshot _(quantitative)_ | kind=flat observable=mean_degree reduce=last | op at_most value 2.0 |


## Study: SBML reactors on a network — diffusive coupling synchronises (`09-sbml-nodes-coupled`)

**Question.** When a continuous SBML oscillator runs at every node and species diffuse along the edges, does the network synchronise?

**Objective.** Run the repressilator_network_coupled composite (SBML reactors + inter-node transport) and measure across-node synchrony.

**Hypothesis.** Diffusive coupling of per-node Repressilators pulls their phases together, so the across-node spread of a protein collapses toward zero — synchronisation.

**Claim.** With an SBML Repressilator at each node and protein diffusion along the edges, the coupled oscillators synchronise — across-node concentration spread collapses.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_dynamic_networks.composites.family_b.repressilator_network_coupled` | 0 | D=0.3, n=9.0, seed=1.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_b.repressilator_network_coupled`**

A ring of Repressilator reactors (SBML at each node, via viva-copasi) coupled by protein diffusion along the edges — the oscillators synchronise.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_b_repressilator_network_coupled = {
    'D'   : 0.3,   # inter-node diffusion coefficient
    'n'   : 9.0,   # number of nodes
    'seed': 1.0,   # phase seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 09-sbml-nodes-coupled ===
STUDY = '09-sbml-nodes-coupled'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_b_repressilator_network_coupled if 'ov_viva_dynamic_networks_composites_family_b_repressilator_network_coupled' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_b_repressilator_network_coupled = materialize_composite('viva_dynamic_networks.composites.family_b.repressilator_network_coupled', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_b_repressilator_network_coupled['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_dynamic_networks.composites.family_b.repressilator_network_coupled) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**concentration-on-network**


In [ ]:
# concentration-on-network
show_viz(_render_one('local:NetworkEvolution', {'title': 'SBML reactors on a network — diffusive coupling synchronises', 'layout': 'circular', 'color_by': 'species:PX', 'subtitle': 'a Repressilator at each node (viva-copasi); proteins diffuse along the ring', 'composite': 'viva_dynamic_networks.composites.family_b.repressilator_network_coupled', 'n_steps': 240, 'render_mode': 'end'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_dynamic_networks_composites_family_b_repressilator_network_coupled if 'ov_viva_dynamic_networks_composites_family_b_repressilator_network_coupled' in dir() else None)))

**concentration animation**


In [ ]:
# concentration animation
show_viz(_render_one('gif:repressilator_network_coupled.gif', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| coupled-synchronises _(quantitative)_ | kind=flat observable=sync_std reduce=last | op at_most value 0.25 |


## Study: SBML reactors on a network — uncoupled control (no diffusion) (`10-sbml-nodes-uncoupled`)

**Question.** With diffusion switched off, do the per-node oscillators stay out of phase?

**Objective.** Run the repressilator_network_uncoupled composite (SBML reactors + inter-node transport) and measure across-node synchrony.

**Hypothesis.** Without inter-node transport each Repressilator runs independently, so nodes seeded at different phases never synchronise and the across-node spread stays large.

**Claim.** With diffusion off, the per-node oscillators remain independent and out of phase — the control that isolates coupling as the cause of synchronisation in study 09.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_dynamic_networks.composites.family_b.repressilator_network_uncoupled` | 0 | n=9.0, seed=1.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_b.repressilator_network_uncoupled`**

The same ring of Repressilator reactors with diffusion off (D=0) — independent oscillators that stay out of phase.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_b_repressilator_network_uncoupled = {
    'n'   : 9.0,   # number of nodes
    'seed': 1.0,   # phase seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 10-sbml-nodes-uncoupled ===
STUDY = '10-sbml-nodes-uncoupled'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_b_repressilator_network_uncoupled if 'ov_viva_dynamic_networks_composites_family_b_repressilator_network_uncoupled' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_b_repressilator_network_uncoupled = materialize_composite('viva_dynamic_networks.composites.family_b.repressilator_network_uncoupled', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_b_repressilator_network_uncoupled['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_dynamic_networks.composites.family_b.repressilator_network_uncoupled) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**concentration-on-network**


In [ ]:
# concentration-on-network
show_viz(_render_one('local:NetworkEvolution', {'title': 'SBML reactors on a network — uncoupled control (no diffusion)', 'layout': 'circular', 'color_by': 'species:PX', 'subtitle': 'identical Repressilator nodes with diffusion off (D=0) — the control', 'composite': 'viva_dynamic_networks.composites.family_b.repressilator_network_uncoupled', 'n_steps': 240, 'render_mode': 'end'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_dynamic_networks_composites_family_b_repressilator_network_uncoupled if 'ov_viva_dynamic_networks_composites_family_b_repressilator_network_uncoupled' in dir() else None)))

**concentration animation**


In [ ]:
# concentration animation
show_viz(_render_one('gif:repressilator_network_uncoupled.gif', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| uncoupled-stays-spread _(quantitative)_ | kind=flat observable=sync_std reduce=last | op at_least value 3.0 |


## Study: Dynamics on a growing network — reactors entrain as the graph is built (`11-coevolving-growing-network`)

**Question.** Can SBML reactor dynamics run on a network whose topology is itself being built by a process-bigraph reaction, and do newcomers entrain?

**Objective.** Run the growing_repressilator_network composite — SBML reactors + transport on a topology that a PBG reaction reconfigures during the run — and measure synchrony.

**Hypothesis.** Running the Family-B reactors + transport on a graph that a Family-A growth reaction is simultaneously building, each newcomer joins at its own phase and is pulled into synchrony by the dense local coupling — the full synthesis of dynamics OF and ON the network.

**Claim.** Continuous SBML dynamics run correctly on a network being built node-by-node by a PBG reaction: the system stays bounded and oscillating, and most newcomers entrain through the dense local coupling — but synchronisability is topology-dependent, with weakly-attached (low-degree) nodes lagging the synchronised core.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_dynamic_networks.composites.family_c.growing_repressilator_network` | 0 | D=0.25, max_nodes=11.0, grow_every=10.0, seed=2.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_c.growing_repressilator_network`**

Dynamics on a GROWING topology: forest-fire growth adds Repressilator reactor-nodes over time while protein diffusion couples them — newcomers join at their own phase and entrain. (Forest-fire growth gives the dense local coupling that lets the growing network synchronise at a numerically stable D.)


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_c_growing_repressilator_network = {
    'D'         : 0.25,   # inter-node diffusion coefficient
    'max_nodes' : 11.0,   # final network size
    'grow_every': 10.0,   # model-time units between node additions
    'seed'      : 1.0,   # phase/RNG seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 11-coevolving-growing-network ===
STUDY = '11-coevolving-growing-network'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_c_growing_repressilator_network if 'ov_viva_dynamic_networks_composites_family_c_growing_repressilator_network' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_c_growing_repressilator_network = materialize_composite('viva_dynamic_networks.composites.family_c.growing_repressilator_network', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_c_growing_repressilator_network['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_dynamic_networks.composites.family_c.growing_repressilator_network) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**concentration-on-coevolving-network**


In [ ]:
# concentration-on-coevolving-network
show_viz(_render_one('local:NetworkEvolution', {'title': 'Dynamics on a growing network — reactors entrain as the graph is built', 'layout': 'spring', 'color_by': 'species:PX', 'subtitle': 'forest-fire growth adds Repressilator nodes under the running reactors; newcomers entrain', 'composite': 'viva_dynamic_networks.composites.family_c.growing_repressilator_network', 'n_steps': 170, 'render_mode': 'end'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_dynamic_networks_composites_family_c_growing_repressilator_network if 'ov_viva_dynamic_networks_composites_family_c_growing_repressilator_network' in dir() else None)))

**concentration animation**


In [ ]:
# concentration animation
show_viz(_render_one('gif:growing_repressilator_network.gif', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| grows-and-mostly-entrains _(quantitative)_ | kind=flat observable=sync_std reduce=last | op at_most value 150.0 |


## Study: Dynamics on a rewiring network — synchrony survives continual rewiring (`12-coevolving-rewiring-network`)

**Question.** Do the coupled oscillators stay synchronised when the very edges that couple them are continuously rewired?

**Objective.** Run the rewiring_repressilator_network composite — SBML reactors + transport on a topology that a PBG reaction reconfigures during the run — and measure synchrony.

**Hypothesis.** With the topology continuously small-world-rewired under the running reactors (node and edge counts conserved), the coupling remains strong enough to reach and hold synchrony.

**Claim.** Coupled SBML oscillators synchronise and stay synchronised even as the coupling edges are continuously rewired — reaction–diffusion is robust to an evolving (but well-connected) topology.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_dynamic_networks.composites.family_c.rewiring_repressilator_network` | 0 | D=0.25, p=0.6, n=12.0, seed=1.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_c.rewiring_repressilator_network`**

Dynamics on a REWIRING topology: a ring of Repressilator reactors whose edges are continuously small-world-rewired while the coupled oscillators run.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_c_rewiring_repressilator_network = {
    'D'   : 0.25,   # inter-node diffusion coefficient
    'p'   : 0.5,   # per-step rewiring probability
    'n'   : 12.0,   # number of nodes
    'seed': 1.0,   # phase/RNG seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 12-coevolving-rewiring-network ===
STUDY = '12-coevolving-rewiring-network'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_c_rewiring_repressilator_network if 'ov_viva_dynamic_networks_composites_family_c_rewiring_repressilator_network' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_c_rewiring_repressilator_network = materialize_composite('viva_dynamic_networks.composites.family_c.rewiring_repressilator_network', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_c_rewiring_repressilator_network['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_dynamic_networks.composites.family_c.rewiring_repressilator_network) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**concentration-on-coevolving-network**


In [ ]:
# concentration-on-coevolving-network
show_viz(_render_one('local:NetworkEvolution', {'title': 'Dynamics on a rewiring network — synchrony survives continual rewiring', 'layout': 'circular', 'color_by': 'species:PX', 'subtitle': 'a ring of Repressilator reactors whose edges are continuously small-world-rewired', 'composite': 'viva_dynamic_networks.composites.family_c.rewiring_repressilator_network', 'n_steps': 170, 'render_mode': 'end'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_dynamic_networks_composites_family_c_rewiring_repressilator_network if 'ov_viva_dynamic_networks_composites_family_c_rewiring_repressilator_network' in dir() else None)))

**concentration animation**


In [ ]:
# concentration animation
show_viz(_render_one('gif:rewiring_repressilator_network.gif', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| rewires-and-synchronises _(quantitative)_ | kind=flat observable=sync_std reduce=last | op at_most value 0.3 |


## Study: Quorum-sensing coupling (proposed) (`13-quorum-sensing-coupling`)

**Question.** Does coupling through a single diffusible signal molecule synchronise the networked reactors as effectively as diffusing the internal proteins?

**Objective.** Extend the node SBML model with a signaling species and diffuse only that molecule; compare synchronisation to study 09.

**Hypothesis.** A dedicated autoinducer species produced and sensed per node, diffusing along the edges, provides indirect coupling sufficient for synchronisation at lower transport flux than direct protein diffusion.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `starting-point` | `viva_dynamic_networks.composites.family_b.repressilator_network_coupled` | 0 | D=0.3, n=9.0, seed=1.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_b.repressilator_network_coupled`**

A ring of Repressilator reactors (SBML at each node, via viva-copasi) coupled by protein diffusion along the edges — the oscillators synchronise.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_b_repressilator_network_coupled = {
    'D'   : 0.3,   # inter-node diffusion coefficient
    'n'   : 9.0,   # number of nodes
    'seed': 1.0,   # phase seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 13-quorum-sensing-coupling ===
STUDY = '13-quorum-sensing-coupling'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_starting_point = 100
INTERVAL_starting_point = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_b_repressilator_network_coupled if 'ov_viva_dynamic_networks_composites_family_b_repressilator_network_coupled' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_b_repressilator_network_coupled = materialize_composite('viva_dynamic_networks.composites.family_b.repressilator_network_coupled', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_b_repressilator_network_coupled['state']}, core=core)
            _comp.run(STEPS_starting_point)
        print('ran', 'starting-point', f'(viva_dynamic_networks.composites.family_b.repressilator_network_coupled) for', STEPS_starting_point, 'steps')
    except Exception as _e:
        print(f"starting-point not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

## Study: Model-agnostic node reactor — Brusselator (proposed) (`14-brusselator-model-agnostic`)

**Question.** Is the SBML-at-nodes architecture model-agnostic — does swapping the Repressilator for a Brusselator reproduce reaction–diffusion patterning on the network with no code change?

**Objective.** Vendor a Brusselator SBML, point the reactors at it, and reproduce a Family-B/C study to demonstrate model-independence.

**Hypothesis.** Because NetworkSBMLReactors loads any SBML model and NetworkTransport diffuses any species, pointing them at a Brusselator yields Turing-like patterning / synchronisation on the network with the identical process code.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `starting-point` | `viva_dynamic_networks.composites.family_c.growing_repressilator_network` | 0 | D=0.25, max_nodes=11.0, seed=2.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_c.growing_repressilator_network`**

Dynamics on a GROWING topology: forest-fire growth adds Repressilator reactor-nodes over time while protein diffusion couples them — newcomers join at their own phase and entrain. (Forest-fire growth gives the dense local coupling that lets the growing network synchronise at a numerically stable D.)


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_c_growing_repressilator_network = {
    'D'         : 0.25,   # inter-node diffusion coefficient
    'max_nodes' : 11.0,   # final network size
    'grow_every': 10.0,   # model-time units between node additions
    'seed'      : 1.0,   # phase/RNG seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 14-brusselator-model-agnostic ===
STUDY = '14-brusselator-model-agnostic'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_starting_point = 100
INTERVAL_starting_point = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_c_growing_repressilator_network if 'ov_viva_dynamic_networks_composites_family_c_growing_repressilator_network' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_c_growing_repressilator_network = materialize_composite('viva_dynamic_networks.composites.family_c.growing_repressilator_network', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_c_growing_repressilator_network['state']}, core=core)
            _comp.run(STEPS_starting_point)
        print('ran', 'starting-point', f'(viva_dynamic_networks.composites.family_c.growing_repressilator_network) for', STEPS_starting_point, 'steps')
    except Exception as _e:
        print(f"starting-point not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

## Study: SBML-driven adaptive coevolution (proposed) (`15-sbml-adaptive-coevolution`)

**Question.** When the continuous node state (an SBML readout) itself drives the rewiring, does a self-organised modular topology emerge?

**Objective.** Couple an SBML-observable-driven rewiring rule to the Family-B reactors, so topology and continuous state co-adapt (the continuous generalisation of study 07).

**Hypothesis.** Letting each node rewire toward neighbours with a similar SBML observable (e.g. oscillator phase) closes the state→topology feedback and self-organises the network into phase-coherent modules.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `starting-point` | `viva_dynamic_networks.composites.family_a.adaptive_rewire` | 0 | w=0.65, n=40.0, seed=0.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_dynamic_networks.composites.family_a.adaptive_rewire`**

Gross–Blasius coevolution: opinions spread voter-style while, with rate w, nodes cut ties to differing neighbours and reconnect to like-minded ones — topology and state co-adapt.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_dynamic_networks_composites_family_a_adaptive_rewire = {
    'w'   : 0.6,   # rewiring rate
    'n'   : 40.0,   # number of nodes
    'k'   : 4.0,   # ring lattice degree
    'seed': 0.0,   # RNG seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: 15-sbml-adaptive-coevolution ===
STUDY = '15-sbml-adaptive-coevolution'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_starting_point = 100
INTERVAL_starting_point = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_dynamic_networks_composites_family_a_adaptive_rewire if 'ov_viva_dynamic_networks_composites_family_a_adaptive_rewire' in dir() else None  # edits from the panel cell, if any
            spec_viva_dynamic_networks_composites_family_a_adaptive_rewire = materialize_composite('viva_dynamic_networks.composites.family_a.adaptive_rewire', _ov)
            _comp = Composite({'state': spec_viva_dynamic_networks_composites_family_a_adaptive_rewire['state']}, core=core)
            _comp.run(STEPS_starting_point)
        print('ran', 'starting-point', f'(viva_dynamic_networks.composites.family_a.adaptive_rewire) for', STEPS_starting_point, 'steps')
    except Exception as _e:
        print(f"starting-point not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

## Open decisions
- Build out the proposed follow-ups (studies 13–15)?
